In [8]:
import time
import re
import gzip
import builtins
import pandas as pd

In [ ]:
class VCFMetaEntry(dict):
    """
    Class for processing VCF metadata entries so they can be used for parsing the main file contents
    """
    
    meta_id = None
    number = None
    desc = None
    type = None

    
    def __init__(self, field):
        self.field = field


def get_read_params(file_path, force_format='infer'):
    """
    Checks the supplied path for file extensions and returns a tibble containing:
    - the module from which .open function will be called (may change it later)
    - the read mode string for that .open function
    """
    
    if force_format == 'vcf':
        return builtins, 'r'

    is_gzip = re.search(r'\.(gz|gzip)$', file_path)

    if is_gzip:
        vcf_file_handler_mod = gzip
        readmode = 'rt'
    else:
        vcf_file_handler_mod = builtins
        readmode = 'r'

    return vcf_file_handler_mod, readmode


def read_header_line(line):
    """
    Reads an unprocessed line from VCF header and returns a VCFHeaderEntry object
    """

    try:
        entry = VCFMetaEntry(re.search(r'##[A-Z]+=', line).group().replace("##", "").replace("=", ""))
        entry.meta_id = re.search(r'ID=([A-Z]|[a-z]|[0-9]|_)+', line).group().replace("ID=", "")
    except AttributeError:
        return None

    try:
        number_val = int(re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", ""))
        entry.number = number_val
    except ValueError:
        number_val = re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", "")
        entry.number = number_val
    except AttributeError:
        entry.number = None

    try:
        type_val = re.search(r'Type=([A-Z]|[a-z]|[0-9])+,', line).group().replace("Type=", "").replace(",", "")
        entry.type = type_val
    except AttributeError:
        entry.type = None

    try:
        entry.desc = re.search(r'Description=".+"', line).group().replace('Description=\"', '').replace('"', '')
    except AttributeError:
        entry.desc = None
    
    return entry

def process_meta_entry(vcf_dict, meta_dict, meta_entry):
    """
    Accepts a VCFMetaEntry object along with 2 dictionaries \
    - first dict is to be used for storing the full vcf data
    - second dict is to be used for storing vcf metadata

    Updates the dictionaries with information from the meta entry
    - adds an {entry field}_{entry id}: [] item to the vcf data dict
    - adds an {entry field}_{entry id}: (entry number, entry value type, entry description)\
    item to the meta dict. Note that meta dict is a nested dictionary \
    (it is a dict with FILTER, FORMAT, and INFO fields, all of which are used\
    to address a dictionary with corresponding meta entry IDs as keys), 
    """
    if meta_entry is not None:
        if meta_entry.field in meta_dict:
            meta_dict[meta_entry.field].update(
                {
                    f'{meta_entry.field}_{meta_entry.meta_id}':\
                    (meta_entry.number, meta_entry.type, meta_entry.desc)
                }
            )
            # print(f'[{meta_entry.field}] {meta_entry.meta_id}, number: {meta_entry.number}, type: {meta_entry.type}, desc: {meta_entry.desc}')
            vcf_dict.update({f'{meta_entry.field}_{meta_entry.meta_id}': []})


def process_filter_entry(vcf_dict, meta_dict, filter_str, filter_delim=';'):
    """
    Accepts a vcf dict and meta dict along with a filter entry string.
    Iterates over FILTER entries in the meta dict.
    If any of the FILTER_{filter entry id} appear in the string, \
    appends the corresponding list in the vcf dict with 1
    Otherwise, appends the corresponding list with 0
    """
    filter_entry_with_prefix = [f'FILTER_{x}' for x in filter_str.split(filter_delim)]
    # print(filter_entry_with_prefix)
    
    for filter_field in meta_dict['FILTER']:
        if filter_field in filter_entry_with_prefix:
            vcf_dict[filter_field].append(1)
        else:
            vcf_dict[filter_field].append(0)


def process_info_entry(vcf_dict, meta_dict, info_str, info_delim=';', info_eq='='):
    """
    Accepts a vcf dict and meta dict along with an info entry string.
    NOTE TO SELF - COULD PROB OPTIMIZE

    Splits the info string based on the supplied delimeter and equal sign
    Iterates over the resulting list and appends lists in the vcf dict with corresponding values
    If any field from vcf meta is not present in the string, \
    appends corresponding list in vcf dict with 0
    """

    info_dict = {}
    for info_item_raw in info_str.split(info_delim):
        info_item = info_item_raw.split(info_eq)
        if len(info_item) == 1:
            info_dict.update({f'INFO_{info_item[0]}': 1})
        elif len(info_item) == 2:
            info_dict.update({f'INFO_{info_item[0]}': info_item[1]})
        else:
            raise IndexError('INFO field entry with wrong syntaxis detected')
    
    info_fields = info_dict.keys()
    for info_field in meta_dict['INFO']:
        if info_field in info_fields:
            vcf_dict[info_field].append(info_dict[info_field])
        else:
            vcf_dict[info_field].append(0)


def process_genotype_entry(vcf_dict, meta_dict, gt_list, format_delim=':'):
    """
    Accepts a vcf dict and meta dict. Unlike other similar function, accepts genotype info \
    as a list of strings.

    Currently only supports a single genotype (FORMAT str + 1 genotype str)

    Iterates over the format fields and appends values to the corresponding vcf dict lists
    """
    format_dict = {
        f'FORMAT_{x[0]}': x[1] for x in
        zip(gt_list.pop(0).split(format_delim), gt_list.pop(0).split(format_delim))
    }

    format_fields = format_dict.keys()
    for format_field in meta_dict['FORMAT']:
        if format_field in format_fields:
            vcf_dict[format_field].append(format_dict[format_field])


def parse_vcf(
    vcf_path,
    filter_delim=';',
    info_delim=';', info_eq='=',
    format_present='infer', format_delim=':',
    line_limit=0, report_freq=100000
):
    """
    
    """
    
    meta_dict = {
        'FILTER': {},
        'INFO': {},
        'FORMAT': {}
    }
    
    vcf_data = {
        'CHROM': [],
        'POS': [],
        'ID': [],
        'REF': [],
        'ALT': [],
        'QUAL': [],
        'INFO': []
    }

    vcf_file_handler_mod, readmode = get_read_params(vcf_path, force_format=format_present)
    
    header_nlines = 0
    with vcf_file_handler_mod.open(vcf_path, readmode) as vcf:
        
        start_time = time.time()
        i = 0
        for i, line in enumerate(vcf):
            if i % report_freq == 0:
                current_time = time.time()
                print(f'Processed {i} entries \
                Elapsed time: {current_time - start_time} seconds.')
            
            # There are 2 major blocks processed differently: the header and the body
            # Currently the algorithm to process a headerless vcf file is not done, so a header is mandatory
    
            if line.startswith('#'):
                header_nlines += 1
                # Both the metadata and the header itself has a # / ## in the beginning
                if line.startswith('##'):
                    # print(line)
                    meta_entry = read_header_line(line)
                    process_meta_entry(vcf_data, meta_dict, meta_entry)
                else:
                    continue        
            else:
                if (i - header_nlines >= line_limit) & (line_limit != 0):
                    # print(vcf_data)
                    # print(meta_dict)
                    break
                
                line_list = line.split('\t')
                # print(line_list)
    
                # First fields are easily processed
                vcf_data['CHROM'].append(line_list.pop(0))
                vcf_data['POS'].append(line_list.pop(0))
                vcf_data['ID'].append(line_list.pop(0))
                vcf_data['REF'].append(line_list.pop(0))
                vcf_data['ALT'].append(line_list.pop(0))
                vcf_data['QUAL'].append(line_list.pop(0))
    
                # FILTER
                # The filter field is referenced to all possible filter field in the metadata
                # If an entry is present, then the column is saved with value 1, else - with 0
                process_filter_entry(
                    vcf_data, meta_dict,
                    filter_str=line_list.pop(0),
                    filter_delim=filter_delim
                )
    
                # INFO
                # Here the program iterates over the possible INFO field values listed in the header too
                # info_entry_with_prefix_raw = [f'INFO_{x}' for x in line_list.pop(0).split(info_delim)]
                process_info_entry(
                    vcf_data, meta_dict,
                    info_str=line_list.pop(0),
                    info_delim=info_delim, info_eq=info_eq
                )

                # FORMAT and genotype columns
                # Currently only a single genotype column is supported
                if line_list == []:
                    continue

                process_genotype_entry(
                    vcf_data, meta_dict,
                    line_list,
                    format_delim=format_delim
                )
                
    
    keys_for_deletion = []
    for vcf_data_key, vcf_data_item in vcf_data.items():
        if len(vcf_data_item) == 0:
            keys_for_deletion.append(vcf_data_key)
    if len(keys_for_deletion) > 0:
        print(f'Warning: items {", ".join(keys_for_deletion)} present in header but not present in data.')
    for key_for_deletion in keys_for_deletion:
        del vcf_data[key_for_deletion]

    vcf_data = pd.DataFrame(vcf_data)

    vcf_data.POS = pd.to_numeric(vcf_data.POS)
    vcf_data.QUAL = pd.to_numeric(vcf_data.QUAL, errors='coerce')
    
    current_time = time.time()
    print(f'Finished processing. Processed {i - header_nlines} entries out of {len(vcf_data)}. Elapsed time: {current_time - start_time} seconds.')
    
    return vcf_data

In [2]:
# Utilities

def get_colnames_after_named_col(vcf_df, col_name='FORMAT'):
    """
    Accepts vcf_df as argument and returns all names of columns after the FORMAT field as a list
    """
    cols_after_named_col = []
    after_named_col = False
    for column in vcf_df.columns:
        if column == col_name:
            after_named_col = True
            continue
        if after_named_col:
            cols_after_named_col.append(column)
    return cols_after_named_col


def iter_joined_combs(iter1, iter2, delim='_'):
    """
    Iterates over the provided 2 iterables and yields each string containing joined iterable items using a provided delimeter
    """
    for i in iter1:
        for j in iter2:
            yield delim.join([i, j])


def get_max_set(series):
    """
    Accepts a series of objects convertible to sets and returns a set with all possible unique items in each iterable object
    """
    max_items = set()
    for itemset in series.apply(lambda x: set(x)):
        max_items.update(itemset)
    return(max_items)


def rename_duplicate_cols(df_to_rename, df_to_compare, suffix='INFO', inplace=False):
    """
    Accepts two dataframes as arguments. Compares column names in df_to_rename to those in df_to_compare.
    If any names overlap, adds a specified suffix to column names in df_to_rename and returns the modified dataframe
    """
    main_df_colnames = df_to_compare.columns
    sec_df_colnames = df_to_rename.columns
    overlapping_names = [name for name in main_df_colnames if name in sec_df_colnames]
    rename_dict = {name: name + suffix for name in overlapping_names}
    if inplace:
        df_to_rename.rename(columns=rename_dict, inplace=True)
    else:
        return df_to_rename.rename(columns=rename_dict)
     
    

In [3]:
# Dealing with VCF header
class VCFMetaEntry:
    """
    Class for processing VCF metadata entries so they can be used for parsing the main file contents
    """
    
    meta_id = None
    number = None
    desc = None
    
    def __init__(self, field):
        self.field = field


def readheader(vcf_file):
    header_fields = []
    nheader = 0
    with open(vcf_path, 'r') as vcf_file:
        for line in vcf_file:
            if line.startswith("##"):
                header_fields.append(line)
                nheader += 1
            else:
                break


def readheaderline(line):
    """
    Reads an unprocessed line from VCF header and returns a VCFHeaderEntry object
    """
    
    entry = VCFMetaEntry(re.search(r'##[A-Z]+=', line).group().replace("##", "").replace("=", ""))
    entry.meta_id = re.search(r'ID=([A-Z]|_)+', line).group().replace("ID=", "")

    try:    
        number_val = int(re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", ""))
        entry.number = number_val
    except ValueError:
        entry.number = number_val
    except AttributeError:
        entry.number = None

    entry.desc = re.search(r'Description=".+"', header_fields[10]).group().replace('Description=\"', '').replace('"', '')
    
    return entry
        
    

In [4]:
# Formatting the vcf_df

def split_format_cols(vcf_df, delim=':'):
    """
    Modifies the input dataframe by splitting the FORMAT column 
    and the following genotype columns using the specified delimeter
    """
    for colname in ['FORMAT'] + get_colnames_after_named_col(vcf_df):
        vcf_df[colname] = vcf_df[colname].str.split(delim)


In [5]:
# Processing the format column

# format_ids = get_max_set(vcf_df.FORMAT)
def generate_formatdict(format_ids, cols_after_format, vcf_row_num):
    return {format_col_name: [None] * vcf_row_num for format_col_name in iter_joined_combs(format_ids, cols_after_format)}
# formatdict = {format_col_name: [None] * vcf_row_num for format_col_name in iter_joined_combs(format_ids, cols_after_format)}

def format_id_val_gen(vcf_df_row_split):
    """
    
    """
    for i, format_id in enumerate(vcf_df_row_split.FORMAT):
        for colname in cols_after_format:
            yield format_id, vcf_df_row_split[colname][i]

def update_formatdict(formatdict, vcf_row, cols_after_format, formatdict_index):
    """
    
    """
    for genotype_name in cols_after_format:
        for format_id, val in format_id_val_gen(vcf_row):
            formatdict[f"{format_id}_{genotype_name}"][formatdict_index] = val

In [6]:
# Processing the info column

def process_infovalue(infovalue, type=None):
    """
    If possible, converts the value of an INFO field from string format to float, int or a list of strings/floats/ints
    """
    if re.match(r'.*,.*', infovalue):
        return [process_infovalue(infoitem) for infoitem in infovalue.split(',')]
    elif re.match(r'[0-9]*\.[0-9]+', infovalue):
        return float(infovalue)
    elif re.match(r'^[0-9]+$', infovalue):
        return int(infovalue)
    else:
        return infovalue
        

def process_info(info, delim=';', equalsign='=', dictlike=True, fieldnames=None, decimal_delim='.'):
    """
    Takes an input INFO field as string and turns it into a dictionary with INFO IDs as keys.
    """
    info = info.split(delim)

    id_regex = re.compile(r'(.+=)|(?!.*=.*)')
    val_regex = re.compile(r'=.+$')
    info_dict = {}
    for infoentry in info:

        try:
            info_id = re.match(id_regex, infoentry).group().replace('=', '')
        except AttributeError:
            raise AttributeError(f'INFO field entry without an ID')
        try:
            infovalue = process_infovalue(re.search(val_regex, infoentry).group().replace('=', ''))
        except AttributeError:
            # We assume that INFO entries with an ID but without a value are flags, and a presence of a flag would set the value to True
            infovalue = True
        # info_dict_entry = {info_id: infovalue}
        info_dict.update({info_id: infovalue})

    return info_dict


def generate_infodict(infoseries, vcf_row_num=None):
    """
    
    """
    if vcf_row_num == None:
        vcf_row_num = len(infoseries)
    first_infoentry = process_info(infoseries[0])
    infodict = {info_id: [None] * vcf_row_num for info_id in first_infoentry.keys()}
    return infodict


def update_infodict(infodict, info, infodict_index):
    """
    Replaces the placeholder values in the <index> position of infodict lists with values from info. The infodict must have lists in all fields. The lists must all have equal length.
    For each key in the info it searches for a matching key in the infodict.
    If the key is present in infodict, it appends the matching infodict list with the value from the info.
    If the key is not in infodict, it first creates a new infoentry in the infodict (filling it with NAs so it matches other lists in the infodict) and then appends it with value from info
    """
    for info_key in info.keys():
        try:
            infodict[info_key][infodict_index] = info[info_key]
        except KeyError:
            infodict_item_len = len(infodict[list(infodict.keys())[0]])
            infodict.update({info_key: [None]*infodict_item_len})
            infodict[info_key][infodict_index] = info[info_key]


def fillinfodict(infodict, infoseries):
    start_time = time.time()
    for i, infoentry in enumerate(infoseries):
        update_infodict(infodict, process_info(infoentry), i)
        if i%100000 == 0:
            current_time = time.time()
            print(f'Processed {i+1} entries out of {vcf_nrow}. Elapsed time: {current_time - start_time} seconds.')
    current_time = time.time()
    print(f'Done processing {vcf_nrow} entries. Elapsed time: {current_time - start_time} seconds.')
    

In [7]:
# Joint processing

def get_infoformatdicts(vcf_df, vocal=False, return_df=False, nrow_limit=0):
    """
    Accepts an vcf loaded as a dataframe and returns a tibble with 2 items:
    - an infodict
    - a formatdict
    !!! IMPORTANT !!! Requires either of 2:
    - no FORMAT and genotype columns in the VCF
    - if FORMAT/genotype columns are present, split_format_cols() must have been applied to the df.
    """
    vcf_row_num = len(vcf_df)
    infodict = generate_infodict(vcf_df.INFO, vcf_row_num)

    try:
        format_present = True
        format_ids = get_max_set(vcf_df.FORMAT)
        cols_after_format = get_colnames_after_named_col(vcf_df, 'FORMAT')
        formatdict = generate_formatdict(format_ids, cols_after_format, vcf_row_num)
    except AttributeError:
        format_present = False
        formatdict = None
        print('Genotype fields not detected, skipping genotype field processing steps')
        
    
    start_time = time.time()
    for i, row in vcf_df.iterrows():
        
        update_infodict(infodict, process_info(row.INFO), i)
        if format_present:
            update_formatdict(formatdict, row, cols_after_format, i)
            
        if i%100000 == 0:
            current_time = time.time()
            print(f'Processed {i+1} entries out of {vcf_row_num}. Elapsed time: {current_time - start_time} seconds.')

        if i + 1 == nrow_limit:
            break

    current_time = time.time()
    print(f'Finished processing. Processed {i+1} entries out of {vcf_row_num}. Elapsed time: {current_time - start_time} seconds.')

    
    if return_df:
        if format_present:
            return (pd.DataFrame(infodict), pd.DataFrame(formatdict))
        else:
            return (pd.DataFrame(infodict), None)
    else:
        return (infodict, formatdict)

In [29]:
# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
vcf_path = f'{wd}/data/test_gridss_output.NA12878.vcf'

In [1]:
# Loading the gridss vcf
vcf_df = pd.read_csv(vcf_path, delimiter='\t', skiprows=137)

NameError: name 'pd' is not defined

In [37]:
def process_vcf_cols(vcf_df):

    try:
        cols_after_format = get_colnames_after_named_col(vcf_df, 'FORMAT')
        split_format_cols(vcf_df)
        format_present = True
    except KeyError:
        format_present = False
        print('Genotype fields not detected, skipping genotype field processing steps')

    infodf, formatdf = get_infoformatdicts(vcf_df, return_df=True)
    
    rename_duplicate_cols(infodf, vcf_df, suffix='INFO', inplace=True)

    cols_to_drop = ['INFO']
    vcf_df = vcf_df.drop(columns = cols_to_drop).join(infodf)
    del infodf
    
    if format_present:
        formatdf = pd.DataFrame(formatdict)
        rename_duplicate_cols(formatdf, vcf_df, suffix='FORMAT', inplace=True)
        cols_to_drop += ['FORMAT'] + cols_after_format
        vcf_df = vcf_df.join(formatdf)
        del formatdf



Object `del` not found.


In [38]:
process_vcf_cols(vcf_df)

Processed 1 entries out of 1007410. Elapsed time: 0.22258257865905762 seconds.
Processed 100001 entries out of 1007410. Elapsed time: 27.41033101081848 seconds.
Processed 200001 entries out of 1007410. Elapsed time: 57.5811653137207 seconds.
Processed 300001 entries out of 1007410. Elapsed time: 84.60774517059326 seconds.
Processed 400001 entries out of 1007410. Elapsed time: 111.46544194221497 seconds.
Processed 500001 entries out of 1007410. Elapsed time: 138.4594211578369 seconds.
Processed 600001 entries out of 1007410. Elapsed time: 165.55556559562683 seconds.
Processed 700001 entries out of 1007410. Elapsed time: 192.64522671699524 seconds.
Processed 800001 entries out of 1007410. Elapsed time: 219.59979462623596 seconds.
Processed 900001 entries out of 1007410. Elapsed time: 246.64234352111816 seconds.
Processed 1000001 entries out of 1007410. Elapsed time: 274.2338635921478 seconds.


In [ ]:
vcf_df

In [10]:
vcf_row_num = len(vcf_df)

In [11]:
# Joint processing (FORMAT + INFO)
format_ids = get_max_set(vcf_df.FORMAT)
cols_after_format = get_colnames_after_named_col(vcf_df, 'FORMAT')
formatdict = generate_formatdict(format_ids, cols_after_format, vcf_row_num)

infodict = generate_infodict(vcf_df.INFO, vcf_row_num)

start_time = time.time()
for i, row in vcf_df.iterrows():
    
    update_infodict(infodict, process_info(row.INFO), i)
    update_formatdict(formatdict, row, cols_after_format, i)
    if i%100000 == 0:
            current_time = time.time()
            print(f'Processed {i+1} entries out of {vcf_row_num}. Elapsed time: {current_time - start_time} seconds.')
    

Processed 1 entries out of 1007410. Elapsed time: 0.3306386470794678 seconds.
Processed 100001 entries out of 1007410. Elapsed time: 28.642284631729126 seconds.
Processed 200001 entries out of 1007410. Elapsed time: 56.3304169178009 seconds.
Processed 300001 entries out of 1007410. Elapsed time: 83.90151357650757 seconds.
Processed 400001 entries out of 1007410. Elapsed time: 111.09189963340759 seconds.
Processed 500001 entries out of 1007410. Elapsed time: 140.63992834091187 seconds.
Processed 600001 entries out of 1007410. Elapsed time: 167.85164260864258 seconds.
Processed 700001 entries out of 1007410. Elapsed time: 195.2122402191162 seconds.
Processed 800001 entries out of 1007410. Elapsed time: 222.42775678634644 seconds.
Processed 900001 entries out of 1007410. Elapsed time: 249.7350013256073 seconds.
Processed 1000001 entries out of 1007410. Elapsed time: 280.18951058387756 seconds.


In [12]:
infodf = pd.DataFrame(infodict)
formatdf = pd.DataFrame(formatdict)

In [13]:
infodf = rename_duplicate_cols(infodf, vcf_df, suffix='INFO')
formatdf = rename_duplicate_cols(formatdf, vcf_df, suffix='FORMAT')

In [14]:
vcf_df = vcf_df.drop(columns = ['INFO', 'FORMAT'] + cols_after_format).join(infodf).join(formatdf)

In [15]:
vcf_df

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,ANRP,ANRPQ,ANSR,...,BSC_NA12878.bwa.bam,BQ_NA12878.bwa.bam,SRQ_NA12878.bwa.bam,BASSR_NA12878.bwa.bam,RP_NA12878.bwa.bam,BSCQ_NA12878.bwa.bam,ANSR_NA12878.bwa.bam,BUM_NA12878.bwa.bam,ASQ_NA12878.bwa.bam,REFPAIR_NA12878.bwa.bam
0,chr1,9952,gridss0bf_1o,N,]chr1:10498]N,498.42,LOW_QUAL;NO_ASSEMBLY,27,408.2,0,...,0,829.69,0.00,0,33,0.00,0,38,0.00,0
1,chr1,10317,gridss0fb_13o,A,ACCCCTTCCCC[chr1:10328[,241.38,LOW_QUAL;SINGLE_ASSEMBLY,0,0.0,0,...,2,35.40,0.00,0,0,35.40,0,0,0.00,500
2,chr1,10325,gridss0fb_14o,C,CCAACCCC[chr1:10333[,230.11,LOW_QUAL;SINGLE_ASSEMBLY,0,0.0,0,...,4,76.37,0.00,0,0,76.37,0,0,0.00,481
3,chr1,10328,gridss0fb_13h,A,]chr1:10317]CCCCTTCCCCA,241.38,LOW_QUAL;SINGLE_ASSEMBLY,0,0.0,0,...,0,0.00,0.00,0,0,0.00,0,0,47.22,482
4,chr1,10332,gridss0f_58b,C,CTTAGCCCTTAGCCCTGGCCCTAGCCCTA.,153.86,LOW_QUAL;NO_RP,0,0.0,0,...,5,153.86,0.00,4,0,85.42,0,0,0.00,452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007405,chrY,12498764,gridss304f_12b,C,CTCCTCCCATCCTCTTCCCTTTCCTCCCCTTCTTCTCCTCCTAACT...,231.00,ASSEMBLY_BIAS;LOW_QUAL;NO_RP;NO_SR,0,0.0,0,...,0,231.00,0.00,6,0,0.00,0,0,0.00,0
1007406,chrY,13754075,gridss1fb_2427h,T,]chr1:13694172]T,209.84,LOW_QUAL;SINGLE_ASSEMBLY,0,0.0,0,...,0,0.00,0.00,0,0,0.00,0,0,0.00,0
1007407,chrY,14707011,gridss283bf_49h,A,AACAC[chr9:90884885[,706.96,SINGLE_ASSEMBLY,0,0.0,0,...,0,0.00,0.00,0,0,0.00,0,0,0.00,0
1007408,chrY,16615430,gridss140ff_9h,A,A]chr2:110221021],59.65,LOW_QUAL;SINGLE_ASSEMBLY,0,0.0,0,...,0,0.00,0.00,0,0,0.00,0,0,0.00,0


In [ ]:
# How to profile python programs - look up myself.
# A profiler will od it for me
# Find hotspots

# Try to make it line by line, not row by row

# Python pickling

# Understand what boosting is. 

# Some stuff for reading in free time
# Hilbert space
# Halting problem
# Turing machines and othe computing models - multiple ones
# Maybe look up types if understandable
